In [2]:
pip install "transformers[torch]"

Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install -U accelerate

In [2]:
import sys

!{sys.executable} -m pip install -U accelerate transformers torch datasets scikit-learn pandas

In [9]:
import accelerate
print(accelerate.__version__)

1.13.0


In [3]:


import pandas as pd
import torch
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

# =========================
# 1. Create Sample Dataset
# =========================

data = {
    "text": [
        "Payment failed twice",
        "Money deducted but order not placed",
        "Refund is still pending",
        "Invoice amount is wrong",
        "Payment gateway error",
        "Refund not processed",
        "Billing issue in my account",

        "Cannot login to my account",
        "Password reset is not working",
        "App is crashing",
        "Website is not opening",
        "Account locked",
        "OTP is not received",
        "Unable to access dashboard",

        "Package damaged",
        "Delivery delayed",
        "Product was broken",
        "Order not delivered",
        "Wrong item delivered",
        "Courier has not arrived",
        "Package lost in delivery"
    ],
    "label": [
        "Billing", "Billing", "Billing", "Billing", "Billing", "Billing", "Billing",
        "Technical", "Technical", "Technical", "Technical", "Technical", "Technical", "Technical",
        "Delivery", "Delivery", "Delivery", "Delivery", "Delivery", "Delivery", "Delivery"
    ]
}

df = pd.DataFrame(data)

# =========================
# 2. Encode Labels
# =========================

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

num_labels = len(label_encoder.classes_)

print("Labels:", label_encoder.classes_)

# =========================
# 3. Train Test Split
# =========================

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

train_dataset = Dataset.from_pandas(train_df[["text", "label_id"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label_id"]])

# =========================
# 4. Tokenization
# =========================

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
test_dataset = test_dataset.rename_column("label_id", "labels")

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# =========================
# 5. Load Model
# =========================

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# =========================
# 6. Metrics
# =========================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions)
    }

# =========================
# 7. Training Arguments
# =========================

training_args = TrainingArguments(
    output_dir="./support_ticket_classifier",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=5,
    report_to="none"
)

# =========================
# 8. Trainer
# =========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# =========================
# 9. Train Model
# =========================

trainer.train()

# =========================
# 10. Evaluate
# =========================

results = trainer.evaluate()
print("Evaluation Results:", results)

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print(classification_report(
    y_true,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

# =========================
# 11. Prediction Function
# =========================

def predict_ticket(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    )

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    return label_encoder.inverse_transform([prediction])[0]

# =========================
# 12. Test New Tickets
# =========================

test_tickets = [
    "Payment failed twice",
    "Cannot login to my account",
    "Package was damaged",
    "Refund is still pending",
    "Delivery is delayed"
]

for ticket in test_tickets:
    print("Ticket:", ticket)
    print("Predicted Label:", predict_ticket(ticket))
    print("-" * 40)

Labels: ['Billing' 'Delivery' 'Technical']


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(l

...

C:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin

Training Loss,Validation Loss,Epoch,Accuracy
0.986252,1.101457,5,0.200000


C:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Evaluation Results: {'eval_loss': 1.1014567613601685, 'eval_accuracy': 0.2}


              precision    recall  f1-score   support

     Billing       0.20      1.00      0.33         1
    Delivery       0.00      0.00      0.00         2
   Technical       0.00      0.00      0.00         2

    accuracy                           0.20         5
   macro avg       0.07      0.33      0.11         5
weighted avg       0.04      0.20      0.07         5

Ticket: Payment failed twice
Predicted Label: Billing
----------------------------------------
Ticket: Cannot login to my account
Predicted Label: Technical
----------------------------------------
Ticket: Package was damaged
Predicted Label: Delivery
----------------------------------------
Ticket: Refund is still pending
Predicted Label: Billing
----------------------------------------
Ticket: Delivery is delayed
Predicted Label: Delivery
----------------------------------------
